# Actividad 2 — Principio de Segregación de Interfaces (ISP)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Dispositivos y Sensores IoT en Cuartos Fríos

---



## 1. Ejemplo Incorrecto (Violando ISP)

Aquí definimos una interfaz enorme `DispositivoFrigorifico(ABC)` que mete en un solo paquete:
- Leer temperatura.
- Leer humedad.
- Modular velocidad del compresor.
- Activar resistencia de descarche (descongelamiento).



In [ ]:
from abc import ABC, abstractmethod

# Interfaz gorda que mezcla sensores y actuadores (Viola ISP)
class DispositivoFrigorificoGordo(ABC):
    @abstractmethod
    def leer_temperatura(self) -> float:
        pass

    @abstractmethod
    def leer_humedad(self) -> float:
        pass

    @abstractmethod
    def encender_compresor(self, rpm: int) -> str:
        pass

    @abstractmethod
    def activar_descarche(self, minutos: int) -> str:
        pass


# Una termocupla simple solo mide temperatura
class TermocuplaSimple(DispositivoFrigorificoGordo):
    def __init__(self, id_sensor: str, valor_temp: float) -> None:
        self.id_sensor: str = id_sensor
        self.valor_temp: float = valor_temp

    def leer_temperatura(self) -> float:
        return self.valor_temp

    # Métodos que no le corresponden, forzados por la interfaz gorda:
    def leer_humedad(self) -> float:
        raise NotImplementedError(f"[{self.id_sensor}] Este sensor NO mide humedad.")

    def encender_compresor(self, rpm: int) -> str:
        raise NotImplementedError(f"[{self.id_sensor}] Este sensor NO es un motor compresor.")

    def activar_descarche(self, minutos: int) -> str:
        raise NotImplementedError(f"[{self.id_sensor}] Este sensor NO tiene resistencias térmicas.")


Interfaz gorda y TermocuplaSimple creadas.


In [2]:
# Demostración del problema con la interfaz gorda
print("--- Probando Interfaz Gorda (Sin ISP) ---")
sensor = TermocuplaSimple("PT100-BODEGA-1", 4.2)

print(f"Temperatura: {sensor.leer_temperatura()}°C")

# Si un controlador general asume que todo dispositivo puede arrancar motores:
try:
    sensor.encender_compresor(2800)
except NotImplementedError as err:
    print(f"Error por culpa de la interfaz gorda: {err}")


--- Probando Interfaz Gorda (Sin ISP) ---
Temperatura: 4.2°C
Error por culpa de la interfaz gorda: [PT100-BODEGA-1] Este sensor NO es un motor compresor.


## 2. Ejemplo Correcto (Aplicando ISP)

Dividimos la interfaz gorda en **interfaces pequeñas y específicas**:
1. `SensorTemperatura(ABC)`: solo para leer temperatura y calibrar.
2. `SensorHumedad(ABC)`: solo para leer humedad y punto de rocío.
3. `ActuadorCompresor(ABC)`: solo para controlar RPM y amperaje del motor.
4. `ControlDescarche(ABC)`: solo para manejar resistencias de descongelamiento.

Cada clase implementa **únicamente lo que de verdad sabe hacer**:
- `TermocuplaIndustrial`: solo implementa `SensorTemperatura`.
- `SensorAmbientalDoble`: implementa `SensorTemperatura` y `SensorHumedad`.
- `CompresorIndustrial`: implementa `ActuadorCompresor` y `ControlDescarche`.


In [ ]:
from abc import ABC, abstractmethod

# 1. Interfaces segregadas y puntuales
class SensorTemperatura(ABC):
    @abstractmethod
    def leer_temperatura_c(self) -> float:
        pass

    @abstractmethod
    def calibrar_sensor(self, offset: float) -> None:
        pass


class SensorHumedad(ABC):
    @abstractmethod
    def leer_humedad_relativa(self) -> float:
        pass

    @abstractmethod
    def calcular_punto_rocio(self) -> float:
        pass


class ActuadorCompresor(ABC):
    @abstractmethod
    def ajustar_velocidad_rpm(self, rpm: int) -> str:
        pass

    @abstractmethod
    def obtener_amperaje(self) -> float:
        pass


class ControlDescarche(ABC):
    @abstractmethod
    def iniciar_descarche_electrico(self, minutos: int) -> str:
        pass

    @abstractmethod
    def estado_descarche(self) -> str:
        pass


# 2. Clases concretas que solo implementan lo que usan
class TermocuplaIndustrial(SensorTemperatura):
    def __init__(self, id_sensor: str, offset: float = 0.0) -> None:
        self.id_sensor: str = id_sensor
        self.offset: float = offset
        self.temp_base: float = 3.5

    def leer_temperatura_c(self) -> float:
        return round(self.temp_base + self.offset, 2)

    def calibrar_sensor(self, offset: float) -> None:
        self.offset = offset


class SensorAmbientalDoble(SensorTemperatura, SensorHumedad):
    def __init__(self, id_sensor: str, temp_inicial: float, humedad_inicial: float) -> None:
        self.id_sensor: str = id_sensor
        self.temp_inicial: float = temp_inicial
        self.humedad_inicial: float = humedad_inicial

    def leer_temperatura_c(self) -> float:
        return self.temp_inicial

    def calibrar_sensor(self, offset: float) -> None:
        self.temp_inicial += offset

    def leer_humedad_relativa(self) -> float:
        return self.humedad_inicial

    def calcular_punto_rocio(self) -> float:
        # fórmula aproximada de punto de rocío
        return round(self.temp_inicial - ((100.0 - self.humedad_inicial) / 5.0), 2)


class CompresorIndustrial(ActuadorCompresor, ControlDescarche):
    def __init__(self, id_motor: str, potencia_hp: float) -> None:
        self.id_motor: str = id_motor
        self.potencia_hp: float = potencia_hp
        self.rpm_actual: int = 0
        self.descarche_prendido: bool = False

    def ajustar_velocidad_rpm(self, rpm: int) -> str:
        self.rpm_actual = rpm
        return f"[{self.id_motor}] Motor ajustado a {self.rpm_actual} RPM"

    def obtener_amperaje(self) -> float:
        return round((self.rpm_actual / 3500.0) * (self.potencia_hp * 3.8), 2)

    def iniciar_descarche_electrico(self, minutos: int) -> str:
        self.descarche_prendido = True
        return f"[{self.id_motor}] Resistencias de descarche encendidas por {minutos} min."

    def estado_descarche(self) -> str:
        return "Descarche Activo" if self.descarche_prendido else "Descarche Apagado"


Interfaces segregadas y clases limpias creadas.


In [4]:
# Clientes que solo consumen lo que necesitan
class MonitorClima:
    def __init__(self, s_temp: SensorTemperatura, s_hum: SensorHumedad) -> None:
        self.s_temp = s_temp
        self.s_hum = s_hum

    def mostrar_reporte(self) -> str:
        t = self.s_temp.leer_temperatura_c()
        h = self.s_hum.leer_humedad_relativa()
        pr = self.s_hum.calcular_punto_rocio()
        return f"Monitor Clima -> Temp: {t}°C | Humedad: {h}% | Punto Rocío: {pr}°C"


class TableroFuerza:
    def __init__(self, motor: ActuadorCompresor) -> None:
        self.motor = motor

    def activar_modo_ahorro(self) -> str:
        msg = self.motor.ajustar_velocidad_rpm(1600)
        amp = self.motor.obtener_amperaje()
        return f"{msg} | Consumo en ahorro: {amp} Amperios"


print("--- Probando Interfaces Segregadas (Cumple ISP) ---\n")

# Instanciamos los dispositivos
termocupla = TermocuplaIndustrial("PT100-SALA-A", offset=-0.2)
sensor_doble = SensorAmbientalDoble("DHT22-SALA-B", temp_inicial=2.8, humedad_inicial=88.0)
compresor = CompresorIndustrial("BITZER-40HP", potencia_hp=40.0)

# El monitor solo pide interfaces de sensórica
monitor = MonitorClima(s_temp=termocupla, s_hum=sensor_doble)
print("1. Reporte Ambiental:")
print(monitor.mostrar_reporte())

# El tablero de fuerza solo pide interfaz de motor
print("\n2. Control de Fuerza y Compresor:")
tablero = TableroFuerza(motor=compresor)
print(tablero.activar_modo_ahorro())
print(compresor.iniciar_descarche_electrico(12))
print(f"Estado: {compresor.estado_descarche()}")


--- Probando Interfaces Segregadas (Cumple ISP) ---

1. Reporte Ambiental:
Monitor Clima -> Temp: 3.3°C | Humedad: 88.0% | Punto Rocío: 0.4°C

2. Control de Fuerza y Compresor:
[BITZER-40HP] Motor ajustado a 1600 RPM | Consumo en ahorro: 69.49 Amperios
[BITZER-40HP] Resistencias de descarche encendidas por 12 min.
Estado: Descarche Activo
